# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an example workflow for loading, exploring, and processing the FAIR² dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant library (if not already installed)
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. Each record set, field, and column is referenced by its `@id`, according to the Croissant specification.

In [ ]:
# List all available record sets and their fields
record_sets = list(dataset.record_sets)
print("Found record sets (by @id):\n")
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"      * Field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")

Below is a peek into the records for one of the record sets (using its `@id`).

In [ ]:
# Show a few records for the first record set
if len(record_sets) > 0:
    example_record_set_id = record_sets[0].id
    print(f"Sample records from record set @id: {example_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:  # Show only first 3 records
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction

Load data from all record sets into pandas DataFrames for analysis. All references use the record set and field `@id` values.

In [ ]:
# Prepare DataFrames for all available record sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set (@id): {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found for record set (@id): {rs_id}")

# For the following EDA, select the first record set with data
selected_rs_id = next((rid for rid in record_set_ids if rid in dataframes and not dataframes[rid].empty), None)
if selected_rs_id:
    print(f"\nUsing record set @id: {selected_rs_id} for further analysis.")
    print(f"Available columns (fields by @id): {dataframes[selected_rs_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field from the record set, filter records, normalize the field, and group by a categorical field. All references below use the appropriate field `@id`.

In [ ]:
# Automatically pick a numeric field for demonstration
import numpy as np

df = dataframes[selected_rs_id] if selected_rs_id else None
if df is not None and not df.empty:
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]

    if not numeric_fields:
        # Try to coerce columns to numeric to find numeric fields
        for c in df.columns:
            coerced = pd.to_numeric(df[c], errors='coerce')
            if not coerced.isna().all():
                df[c] = coerced
        numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]

    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use the first detected numeric field
        print(f"Selected numeric field for EDA: {numeric_field}")
        
        # Filter for values greater than a low threshold (median is chosen for example)
        threshold = df[numeric_field].median() if pd.notna(df[numeric_field].median()) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head(3))

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))

        # Try grouping by another (non-numeric) field if available
        possible_group_fields = [col for col in df.columns if col != numeric_field and not np.issubdtype(df[col].dtype, np.number)]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field} by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields available for EDA in the selected record set.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization

Here we visualize the distribution of the chosen numeric field. More complex visualizations (scatter, boxplot, etc.) can be created depending on data richness.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No visualization possible (no numeric field detected).")

## 6. Conclusion

This notebook demonstrated loading, overview, and manipulation of the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** FAIR² dataset using the `mlcroissant` library.

**Key takeaways:**
- You can access dataset structure via record set and field `@id`s.
- Data can be filtered, normalized, and summarized easily.
- The Croissant schema provides rich metadata for programmatic exploration and analysis.

Refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) for additional advanced usage.